In [ ]:
"""
SIMPLE ALPHA TESTING FRAMEWORK
================================
No fancy classes or interfaces. Just:
1. Load your CSV data
2. Define your alpha function
3. Run backtest
4. See results

For Quant Invitational - quick and dirty testing
"""

import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
def backtest_alpha(df, stock_name, signals, alpha_name="Alpha", initial_capital=100000, transaction_cost=0.001):
    """
    Backtest an alpha strategy
    
    Args:
        df: DataFrame with OHLCV data (must have 'Close' column)
        signals: pandas Series with same index as df, values in [-1,1]
                 1 = long, -1 = short, and by how much
        alpha_name: name of your alpha (for printing)
        initial_capital: starting portfolio value
        transaction_cost: % cost per trade (e.g., 0.001 = 0.1%)
    
    Returns:
        Dictionary with results and metrics
    """
    
    bt = df.copy()
    bt['Signal'] = signals #Adds singal column
    bt['Daily_Return'] = bt[stock_name].pct_change()#caluclate close percent change
    
    # Hold position until signal changes (fixed for pandas 3.0+)
    bt['Position'] = bt['Signal'].ffill().fillna(0)#forward fills
    
    # Transaction cost when position changes
    bt['Position_Change'] = bt['Position'].diff().abs()#calculate the position change, how much we hold has changed
    bt['Trans_Cost'] = bt['Position_Change'] * transaction_cost#the absolute difference is our transaction cost
    
    # Strategy return = signal * daily_return - transaction costs
    bt['Strat_Return'] = bt['Position'].shift(1) * bt['Daily_Return'] - bt['Trans_Cost']
    
    # Cumulative return
    bt['Cumul_Return'] = (1 + bt['Strat_Return']/100).cumprod() - 1
    bt['Portfolio_Value'] = initial_capital * (1 + bt['Cumul_Return'])
    
    # Buy & Hold comparison
    bt['BH_Return'] = bt['Daily_Return']
    bt['BH_Cumul'] = (1 + bt['BH_Return']).cumprod() - 1
    bt['BH_Portfolio'] = initial_capital * (1 + bt['BH_Cumul'])
    
    # Calculate metrics
    metrics = _calculate_metrics(bt, alpha_name, initial_capital)
    
    return {
        'backtest_df': bt,
        'metrics': metrics
    }

In [ ]:
def _calculate_metrics(bt, alpha_name, initial_capital):
    """Calculate performance metrics"""
    
    strat_ret = bt['Strat_Return'].dropna()
    bh_ret = bt['BH_Return'].dropna()
    
    # Returns
    total_return_pct = (bt.iloc[-1]['Portfolio_Value'] / initial_capital - 1) * 100
    bh_total_pct = (bt.iloc[-1]['BH_Portfolio'] / initial_capital - 1) * 100
    excess_return_pct = total_return_pct - bh_total_pct
    
    # Volatility (annualized)
    annual_vol = strat_ret.std() * np.sqrt(252) * 100
    bh_vol = bh_ret.std() * np.sqrt(252) * 100
    
    # Sharpe Ratio (annualized, risk-free rate = 0)
    sharpe = (strat_ret.mean() * 252) / (strat_ret.std() * np.sqrt(252)) if strat_ret.std() > 0 else 0
    bh_sharpe = (bh_ret.mean() * 252) / (bh_ret.std() * np.sqrt(252)) if bh_ret.std() > 0 else 0
    
    # Sortino Ratio (uses only downside volatility)
    # Downside deviation: standard deviation of negative returns only
    negative_returns = strat_ret[strat_ret < 0]
    downside_deviation = negative_returns.std() * np.sqrt(252) if len(negative_returns) > 0 else 0.0001
    
    if downside_deviation > 0:
        sortino = (strat_ret.mean() * 252) / downside_deviation
    else:
        sortino = 0
    
    # Sortino for Buy & Hold
    bh_negative_returns = bh_ret[bh_ret < 0]
    bh_downside_deviation = bh_negative_returns.std() * np.sqrt(252) if len(bh_negative_returns) > 0 else 0.0001
    
    if bh_downside_deviation > 0:
        bh_sortino = (bh_ret.mean() * 252) / bh_downside_deviation
    else:
        bh_sortino = 0

    # Max Drawdown
    cummax = bt['Portfolio_Value'].cummax()
    drawdown = (bt['Portfolio_Value'] - cummax) / cummax
    max_dd = drawdown.min() * 100
    
    # Win Rate
    winning = (strat_ret > 0).sum()
    losing = (strat_ret < 0).sum()
    win_rate = (winning / (winning + losing) * 100) if (winning + losing) > 0 else 0
    
    # Number of trades
    num_trades = int(bt['Position_Change'].sum())
    
    return {
        'Alpha_Name': alpha_name,
        'Total_Return_%': total_return_pct,
        'BH_Return_%': bh_total_pct,
        'Excess_Return_%': excess_return_pct,
        'Annual_Volatility_%': annual_vol,
        'Sharpe_Ratio': sharpe,
        'BH_Sharpe': bh_sharpe,
        'Sortino_Ratio': sortino,         
        'BH_Sortino': bh_sortino,            
        'Max_Drawdown_%': max_dd,
        'Win_Rate_%': win_rate,
        'Num_Trades': num_trades,
        'Final_Value': bt.iloc[-1]['Portfolio_Value'],
        'BH_Final_Value': bt.iloc[-1]['BH_Portfolio']
    }

In [ ]:
def compare_alphas(df,stock_name, alpha_functions_dict, initial_capital=100000):
    """
    Test multiple alphas and compare them
    
    Args:
        df: DataFrame with OHLCV
        alpha_functions_dict: dict like {'Alpha_Name': alpha_function}
                             where alpha_function(df) returns signals
        initial_capital: starting capital
    
    Returns:
        DataFrame with comparison results
    """
    
    results_list = []
    
    for alpha_name, alpha_func in alpha_functions_dict.items():
        print(f"\nTesting {alpha_name}...")
        
        # Generate signals
        signals = alpha_func(df)
        
        # Backtest
        result = backtest_alpha(df, stock_name, signals, alpha_name, initial_capital)
        metrics = result['metrics']
        
        # Print results
        print_results(metrics)
        
        results_list.append(metrics)
    
    # Create comparison table
    comparison_df = pd.DataFrame(results_list)
    
    # Print summary
    print("\n" + "="*70)
    print("COMPARISON SUMMARY - Ranked by Excess Return")
    print("="*70)
    comparison_df_sorted = comparison_df.sort_values('Excess_Return_%', ascending=False)
    
    cols = ['Alpha_Name', 'Excess_Return_%', 'Sharpe_Ratio', 'Max_Drawdown_%', 'Win_Rate_%', 'Num_Trades']
    print(comparison_df_sorted[cols].to_string(index=False))
    
    return comparison_df

In [ ]:
def print_results(metrics):
    """Pretty print backtest results"""
    
    print("\n" + "="*70)
    print(f"ALPHA: {metrics['Alpha_Name']}")
    print("="*70)
    print(f"{'Metric':<35} {'Strategy':>15} {'Buy&Hold':>15}")
    print("-"*70)
    print(f"{'Total Return':<35} {metrics['Total_Return_%']:>14.2f}% {metrics['BH_Return_%']:>14.2f}%")
    print(f"{'Excess Return':<35} {metrics['Excess_Return_%']:>14.2f}% {'-':>15}")
    print(f"{'Annual Volatility':<35} {metrics['Annual_Volatility_%']:>14.2f}% {'-':>15}")
    print(f"{'Sharpe Ratio':<35} {metrics['Sharpe_Ratio']:>14.2f} {metrics['BH_Sharpe']:>14.2f}")
    print(f"{'Sortino Ratio':<35} {metrics['Sortino_Ratio']:>14.2f} {metrics['BH_Sortino']:>14.2f}")
    print(f"{'Max Drawdown':<35} {metrics['Max_Drawdown_%']:>14.2f}% {'-':>15}")
    print(f"{'Win Rate':<35} {metrics['Win_Rate_%']:>14.2f}% {'-':>15}")
    print(f"{'Number of Trades':<35} {metrics['Num_Trades']:>14.0f} {'-':>15}")
    print(f"{'Final Portfolio Value':<35} ${metrics['Final_Value']:>13,.0f} ${metrics['BH_Final_Value']:>13,.0f}")
    print("="*70)


In [ ]:
##New Alpha
def calculate_alpha(df):
    A=np.ones(len(df))
    A=A*0.5
    return A


In [ ]:
def CCC_BBB_Corr(df):
    CCC = df.CCC.values
    BBB = df.BBB.values

    res = np.zeros(len(df))

    for i in range(5, len(df)):
        res[i] = BBB[i - 4:i + 1] - CCC[i - 4:i + 1]

    lar = max(res.max, abs(res.min))

    res /= lar

    return lar

In [ ]:
if __name__ == "__main__":
    
    # ===== STEP 1: LOAD YOUR DATA =====
    print("Loading data...")
    
    csv_file = "data.csv"  # CHANGE THIS TO YOUR FILE
    

    df = pd.read_csv(csv_file)
    
    print(f"✓ Loaded {len(df)} rows")
    print(f"  Date range: {df.index[0]} to {df.index[-1]}")
    print(f"  Columns: {list(df.columns)}")
    
    
    # ===== STEP 2: TEST INDIVIDUAL ALPHAS =====
    print("\n" + "="*70)
    print("TESTING INDIVIDUAL ALPHAS")
    print("="*70)
    
    # Test one alpha
    print("\n--- Testing Momentum Alpha ---")
    signals = CCC_BBB_Corr(df)
    result = backtest_alpha(df, signals, "calculate_alpha")

    print_results(result['metrics'])
    
    # ===== STEP 3: COMPARE MULTIPLE ALPHAS =====
    print("\n" + "="*70)
    print("COMPARING MULTIPLE ALPHAS")
    print("="*70)
    """
    alphas_to_test = {
        'Momentum_20d': lambda df: momentum_alpha(df, lookback=20),
        'Momentum_10d': lambda df: momentum_alpha(df, lookback=10),
        'MeanReversion_20': lambda df: mean_reversion_alpha(df, window=20),
        'MACD': lambda df: macd_alpha(df),
        'RSI': lambda df: rsi_alpha(df),
        'SMA_20_50': lambda df: sma_crossover_alpha(df, fast=20, slow=50),
        'Volume': lambda df: volume_alpha(df),
    }"""
    alphas_to_test={"calculate_alpha":lambda df: calculate_alpha(df)}
    
    comparison = compare_alphas(df, "BBB",alphas_to_test)
    
    # Save comparison
    # Just save in your current folder
    comparison.to_csv('alpha_comparison.csv', index=False)
    
    print("\n✓ Comparison saved to: alpha_comparison.csv")

Loading data...
Creating sample data (file ./Data Analys/data.csv not found)...

TESTING INDIVIDUAL ALPHAS

--- Testing Momentum Alpha ---


AttributeError: 'DataFrame' object has no attribute 'CCC'